# History-Aware, RAG-backed Chatbot

## 1. RAG

In [17]:
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# Initialize the embedding model that should match the one used for the existing db
embedding_function = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Load the existing db from the local folder
vector_store  = Chroma(
    persist_directory="./chroma_db",
    collection_name="my_rag_db",
    embedding_function=embedding_function,
)

# convert the vector store to a seatch retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4670.82it/s]


In [18]:
# test retriever
response = retriever.invoke("What's effective intensity?")
print(response)

[Document(id='85f1fb95-d9f6-43a7-8c79-37aad8ecf2f9', metadata={'source': 'context_files/README.rst'}, page_content='Effective luminous intensity is a concept used in photometry to\nquantify the perceived brightness of a light source, particularly in\nthe context of flashing lights or other time-varying light sources, which\nare commonly used in signaling applications such as aviation, marine\nnavigation, and land transportation.\nIt represents the intensity (also called effective intensity, measured by cd) \nof a steady light source that would appear equally bright to the human eye as'), Document(id='dfb262ac-4eaf-4fcb-a057-79cfa9a9adc9', metadata={'source': 'context_files/README.rst'}, page_content='Effective luminous intensity is a concept used in photometry to\nquantify the perceived brightness of a light source, particularly in\nthe context of flashing lights or other time-varying light sources, which\nare commonly used in signaling applications such as aviation, marine\nnavigation

# 2. LLM

In [2]:
from langchain_groq import ChatGroq
from decouple import Config, RepositoryEnv
env_config = Config(RepositoryEnv("./.env"))

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    # model="llama-3.1-8b-instant",
    temperature=1.0,
    max_retries=2,
    api_key=env_config("GROQ_API_KEY"),
)

## 3. History-aware RAG

In [19]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever

# create a prompt template for the retriever to use, which includes the retrieved documents and the user query
contextualized_system_prompt = "You are a helpful assistant. Use the following retrieved documents to answer the question. If you don't know the answer, say you don't know. Always use all the retrieved documents and never ignore any of them. Always use the exact same information in the retrieved documents without adding any additional information. If there is a conflict between the retrieved documents, use all the conflicting information and never ignore any of them."

# the retriever will fill in the retrieved documents into the messages placeholder, and the user query will be filled in the user message
contextulized_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualized_system_prompt),
    messages_placeholder := MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}")
])

# create a history aware retriever chain, which will manage the conversation history and pass it to the retriever and the llm
history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextulized_prompt
)

# test retriever
response = history_aware_retriever.invoke({"input": "What's effective intensity?", "chat_history": []})
print(response)

[Document(id='85f1fb95-d9f6-43a7-8c79-37aad8ecf2f9', metadata={'source': 'context_files/README.rst'}, page_content='Effective luminous intensity is a concept used in photometry to\nquantify the perceived brightness of a light source, particularly in\nthe context of flashing lights or other time-varying light sources, which\nare commonly used in signaling applications such as aviation, marine\nnavigation, and land transportation.\nIt represents the intensity (also called effective intensity, measured by cd) \nof a steady light source that would appear equally bright to the human eye as'), Document(id='dfb262ac-4eaf-4fcb-a057-79cfa9a9adc9', metadata={'source': 'context_files/README.rst'}, page_content='Effective luminous intensity is a concept used in photometry to\nquantify the perceived brightness of a light source, particularly in\nthe context of flashing lights or other time-varying light sources, which\nare commonly used in signaling applications such as aviation, marine\nnavigation

## 4. Chain

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the following retrieved documents to answer the question. If you don't know the answer, say you don't know. Always use all the retrieved documents and never ignore any of them. Always use the exact same information in the retrieved documents without adding any additional information. If there is a conflict between the retrieved documents, use all the conflicting information and never ignore any of them."),
    ("system", "Context: {context}"),
    ("user", "{input}")
])

# create a chain to combine the retrieved documents and the user query to generate the final answer
answer_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=template,
)

# rag_chain = history_aware_retriever | answer_chain
rag_chain = create_retrieval_chain(
    history_aware_retriever, 
    answer_chain
)


## 5. Add Chat Memory - Optional

In [21]:
from langchain_core.messages import AIMessage, HumanMessage

chat_history = []

query1 = "Hi I'am John. What is effective intensity?"
res1 = rag_chain.invoke({"input": query1, "chat_history": chat_history})["answer"]
print("Answer: ", res1)
chat_history.extend([
    HumanMessage(content=query1), 
    AIMessage(content=res1)
])

query2 = "Who invented it?"
res2 = rag_chain.invoke({"input": query2, "chat_history": chat_history})["answer"]
print("Answer: ", res2)
chat_history.extend([
    HumanMessage(content=query2), 
    AIMessage(content=res2)
])

Answer:  Hello John, according to the documents, effective intensity, also called effective luminous intensity, is a concept used in photometry to quantify the perceived brightness of a light source. It represents the intensity of a steady light source that would appear equally bright to the human eye, and it is measured in candela (cd).
Answer:  I don't know. The provided documents do not mention who invented the concept of effective luminous intensity.


## 6. Multi-User Chatbot

In [29]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Maintain an in-memory dictionary to store distinct session tracks
session_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

chain_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",           # Maps to the 'human' prompt key
    history_messages_key="chat_history",  # Maps to the 'MessagesPlaceholder' key
    output_messages_key="answer"           # Maps to the 'ai' response key
)

In [31]:
# test
config = {
    "configurable": {
        "session_id": "user1"
    }
}

query1 = "Hi I'am John. What is effective intensity?"
res1 = chain_with_history.invoke({"input": query1}, config=config).get("answer")
print("Answer: ", res1)

query2 = "Who invented it?"
res2 = chain_with_history.invoke({"input": query2}, config=config).get("answer")
print("Answer: ", res2)

Answer:  Hello John, effective intensity is a concept used in photometry to quantify the perceived brightness of a light source. It represents the intensity of a steady light source that would appear equally bright to the human eye, and it's measured by cd (candela). This concept is particularly used in the context of flashing lights or other time-varying light sources, commonly found in signaling applications such as aviation, marine navigation, and land transportation.
Answer:  I don't know. The provided documents do not mention who invented the concept of effective luminous intensity.


## SQLite3